### Imports

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [46]:
import pandas as pd
import numpy as np
import re

from transformers import pipeline, AutoTokenizer, AutoModel
import torch

### CSV-Daten laden

Donald Trump Tweets bis 2021

In [3]:
tweets = pd.read_csv("C:/Users/02_Florian_Benutzer/Desktop/trump_oil_analysis_v2/00_data/trump_tweets.csv")

In [4]:
tweets.head()

,id,text,is_retweet,is_deleted,device,favorites,retweets,datetime,is_flagged,date
0,9.845497e+16,Republicans and Democrats have both created ou...,False,False,TweetDeck,49,255,2011-08-02T18:07:48Z,False,2011-08-02
1,1.234653e+18,I was thrilled to be back in the Great city of...,False,False,Twitter for iPhone,73748,17404,2020-03-03T01:34:50Z,False,2020-03-03
2,1.218011e+18,RT @CBS_Herridge: READ: Letter to surveillance...,True,False,Twitter for iPhone,0,7396,2020-01-17T03:22:47Z,False,2020-01-17
3,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,False,False,Twitter for iPhone,80527,23502,2020-09-12T20:10:58Z,False,2020-09-12
4,1.218160e+18,RT @MZHemingway: Very friendly telling of even...,True,False,Twitter for iPhone,0,9081,2020-01-17T13:13:59Z,False,2020-01-17


In [5]:
tweets.shape

(56571, 10)

In [6]:
tweets_v1 = tweets.copy()

In [7]:
# 2. Den String in ein "naives" (zeitzonenloses) Datetime-Objekt umwandeln
# (Pandas erkennt das Format meistens automatisch)
tweets_v1['timestamp_utc'] = pd.to_datetime(tweets_v1['datetime'])

# 4. In UTC umrechnen
#tweets_v1['timestamp_utc'] = tweets_v1['timestamp'].dt.tz_localize('UTC')

# 5. (Optional) Wenn du das "+00:00" am Ende der UTC-Zeit nicht in der CSV haben willst,
# kannst du die Zeitzonen-Information wieder entfernen (die Zeit bleibt die von UTC):
#tweets_v1['timestamp_utc_clean'] = tweets_v1['timestamp_utc'].dt.tz_localize(None)

# Ergebnis überprüfen
#print(df[['Timestamp', 'Timestamp_UTC_clean']].head())

In [8]:
tweets_v1.dtypes

id                           float64
text                             str
is_retweet                      bool
is_deleted                      bool
device                           str
favorites                      int64
retweets                       int64
datetime                         str
is_flagged                      bool
date                             str
timestamp_utc    datetime64[us, UTC]
dtype: object

In [9]:
tweets_v1.shape

(56571, 11)

In [10]:
tweets_v1.head()

,id,text,is_retweet,is_deleted,device,favorites,retweets,datetime,is_flagged,date,timestamp_utc
0,9.845497e+16,Republicans and Democrats have both created ou...,False,False,TweetDeck,49,255,2011-08-02T18:07:48Z,False,2011-08-02,2011-08-02 18:07:48+00:00
1,1.234653e+18,I was thrilled to be back in the Great city of...,False,False,Twitter for iPhone,73748,17404,2020-03-03T01:34:50Z,False,2020-03-03,2020-03-03 01:34:50+00:00
2,1.218011e+18,RT @CBS_Herridge: READ: Letter to surveillance...,True,False,Twitter for iPhone,0,7396,2020-01-17T03:22:47Z,False,2020-01-17,2020-01-17 03:22:47+00:00
3,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,False,False,Twitter for iPhone,80527,23502,2020-09-12T20:10:58Z,False,2020-09-12,2020-09-12 20:10:58+00:00
4,1.218160e+18,RT @MZHemingway: Very friendly telling of even...,True,False,Twitter for iPhone,0,9081,2020-01-17T13:13:59Z,False,2020-01-17,2020-01-17 13:13:59+00:00


Retweets werden aus der Analyse ausgeschlossen

In [11]:
tweets_v2 = tweets_v1.copy()

In [12]:
tweets_v2 = tweets_v2[tweets_v2['is_retweet'] == False] 

In [13]:
tweets_v2.shape

(46694, 11)

Die Spalten 'is_deleted', 'favorites', 'retweets' und 'is_flagged' werden entfernt um Data Leakage zu vermeiden

In [14]:
tweets_v2 = tweets_v2.drop(columns=['is_retweet', 'is_deleted', 'device', 'favorites', 'retweets', 'is_flagged'])

In [15]:
tweets_v2.shape

(46694, 5)

In [16]:
tweets_v2

,id,text,datetime,date,timestamp_utc
0,9.845497e+16,Republicans and Democrats have both created ou...,2011-08-02T18:07:48Z,2011-08-02,2011-08-02 18:07:48+00:00
1,1.234653e+18,I was thrilled to be back in the Great city of...,2020-03-03T01:34:50Z,2020-03-03,2020-03-03 01:34:50+00:00
3,1.304875e+18,The Unsolicited Mail In Ballot Scam is a major...,2020-09-12T20:10:58Z,2020-09-12,2020-09-12 20:10:58+00:00
6,1.223641e+18,Getting a little exercise this morning! https:...,2020-02-01T16:14:02Z,2020-02-01,2020-02-01 16:14:02+00:00
7,1.319502e+18,https://t.co/4qwCKQOiOw,2020-10-23T04:52:14Z,2020-10-23,2020-10-23 04:52:14+00:00
...,...,...,...,...,...
56555,1.213079e+18,"Iran never won a war, but never lost a negotia...",2020-01-03T12:44:30Z,2020-01-03,2020-01-03 12:44:30+00:00
56559,1.212177e+18,Thank you to the @dcexaminer Washington Examin...,2020-01-01T01:03:15Z,2020-01-01,2020-01-01 01:03:15+00:00
56560,1.212175e+18,One of my greatest honors was to have gotten C...,2020-01-01T00:55:01Z,2020-01-01,2020-01-01 00:55:01+00:00
56569,1.319384e+18,Just signed an order to support the workers of...,2020-10-22T21:04:21Z,2020-10-22,2020-10-22 21:04:21+00:00


In [17]:
tweets_v2.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v2.csv", sep=",", index=False, encoding="utf-8")

In [24]:
df = tweets_v2.copy()

## Basic Cleaning

In [27]:
def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)   # URLs
    text = re.sub(r"@\w+", "", text)      # mentions
    text = re.sub(r"#", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].astype(str).apply(clean_tweet)

## Temporal Features

In [31]:
# df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

df["hour"] = df["timestamp_utc"].dt.hour
df["minute"] = df["timestamp_utc"].dt.minute
df["day_of_week"] = df["timestamp_utc"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

## Rule-Based NLP Features

In [ ]:
# OIL_TERMS = ["oil", "barrel", "opec", "gas", "petroleum", "energy"]
# TRADE_TERMS = ["china", "tariff", "trade", "deal"]
# GEO_TERMS = ["iran", "syria", "russia", "war", "military"]
# ECON_TERMS = ["fed", "interest", "inflation", "jobs", "gdp"]

In [35]:
OIL_TERMS = [
    "oil", "crude", "crude oil", "wti", "brent", "barrel",
    "opec", "opec+", "gas", "natural gas", "lng",
    "petroleum", "energy",
    "refinery", "refining", "drilling", "rig", "rigs",
    "production", "output", "supply", "demand",
    "inventories", "stockpile", "storage",
    "spr", "strategic petroleum reserve",
    "inventory", "exports", "imports",
    "futures", "commodity", "commodities"
]

TRADE_TERMS = [
    "china", "tariff", "trade", "deal", "agreement",
    "tariffs", "sanctions", "embargo",
    "nafta", "usmca", "eu", "europe",
    "mexico", "canada",
    "trade war", "currency manipulation"
]

GEO_TERMS = [
    "iran", "iraq", "syria", "russia", "venezuela",
    "saudi", "saudi arabia", "libya",
    "war", "military", "conflict",
    "attack", "strike", "bombing",
    "middle east", "geopolitical"
]

ECON_TERMS = [
    "fed", "federal reserve", "interest", "interest rate",
    "inflation", "cpi", "pce",
    "jobs", "unemployment", "nonfarm payrolls", "nfp",
    "gdp", "growth",
    "recession",
    "dollar", "usd",
    "yield", "treasury",
    "quantitative easing", "qe",
    "monetary policy",
    "stimulus", "fiscal", "budget"
]

# Optional (sehr relevant für Trump-Tweets als Kontextsignal)
# TRUMP_CONTEXT_TERMS = [
#     "trump", "donald trump", "president", "white house",
#     "administration"
# ]

In [36]:
def keyword_flag(text, keywords):
    return int(any(k in text for k in keywords))


df["mentions_oil"] = df["clean_text"].apply(lambda x: keyword_flag(x, OIL_TERMS))
df["mentions_trade"] = df["clean_text"].apply(lambda x: keyword_flag(x, TRADE_TERMS))
df["mentions_geo"] = df["clean_text"].apply(lambda x: keyword_flag(x, GEO_TERMS))
df["mentions_econ"] = df["clean_text"].apply(lambda x: keyword_flag(x, ECON_TERMS))

## Stylometric Features (Trump Signatur)

In [38]:
df["tweet_length"] = df["clean_text"].str.len()
df["word_count"] = df["clean_text"].str.split().str.len()

df["exclamation_count"] = df["text"].str.count("!")
df["question_count"] = df["text"].str.count(r"\?")

df["caps_ratio"] = df["text"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1)
)

## Sentiment FinBERT

In [47]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [48]:
def get_sentiment(text):
    try:
        res = sentiment_model(text[:512])[0]
        label = res["label"]
        score = res["score"]

        if label == "positive":
            return score
        elif label == "negative":
            return -score
        else:
            return 0
    except:
        return 0


df["sentiment_score"] = df["clean_text"].apply(get_sentiment)

## Emotion Proxy

In [49]:
POS_WORDS = ["great", "strong", "win", "good", "amazing"]
NEG_WORDS = ["bad", "terrible", "weak", "crisis", "fake"]

def emotion_score(text):
    pos = sum(w in text for w in POS_WORDS)
    neg = sum(w in text for w in NEG_WORDS)
    return pos - neg


df["emotion_score"] = df["clean_text"].apply(emotion_score)

## Transformer Embedding (Sentence-BERT)

In [51]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\02_Florian_Benutzer\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [52]:
embeddings = embed_model.encode(
    df["clean_text"].tolist(),
    show_progress_bar=True
)

embeddings_df = pd.DataFrame(
    embeddings,
    columns=[f"emb_{i}" for i in range(embeddings.shape[1])]
)

df = pd.concat([df.reset_index(drop=True), embeddings_df], axis=1)

Batches:   0%|          | 0/1460 [00:00<?, ?it/s]

## Optional: PCA reduzieren (empfohlen)

In [53]:
from sklearn.decomposition import PCA

pca = PCA(n_components=30)
emb_pca = pca.fit_transform(embeddings)

emb_pca_df = pd.DataFrame(
    emb_pca,
    columns=[f"emb_pca_{i}" for i in range(30)]
)

df = pd.concat([df, emb_pca_df], axis=1)

## Burst Features

In [54]:
df = df.sort_values("timestamp_utc")

df["tweets_last_1h"] = df["timestamp_utc"].rolling("1h", on="timestamp_utc").count()
df["tweets_last_6h"] = df["timestamp_utc"].rolling("6h", on="timestamp_utc").count()

ValueError: invalid on specified as timestamp_utc, must be a column (of DataFrame), an Index or None

### Alternative

In [57]:
df.set_index("timestamp_utc", inplace=True)

df["tweets_last_1h"] = df["text"].rolling("1h").count()
df["tweets_last_6h"] = df["text"].rolling("6h").count()

df = df.reset_index()

In [58]:
df

,timestamp_utc,id,text,datetime,date,clean_text,hour,day_of_week,is_weekend,minute,...,emb_pca_22,emb_pca_23,emb_pca_24,emb_pca_25,emb_pca_26,emb_pca_27,emb_pca_28,emb_pca_29,tweets_last_1h,tweets_last_6h
0,2009-05-04 18:54:25+00:00,1.698309e+09,Be sure to tune in and watch Donald Trump on L...,2009-05-04T18:54:25Z,2009-05-04,be sure to tune in and watch donald trump on l...,18,0,0,54,...,0.147628,-0.120222,-0.003056,-0.042803,0.012091,0.036382,-0.011314,0.060289,1.0,1.0
1,2009-05-05 01:00:10+00:00,1.701461e+09,Donald Trump will be appearing on The View tom...,2009-05-05T01:00:10Z,2009-05-05,donald trump will be appearing on the view tom...,1,1,0,0,...,0.115513,-0.006284,-0.036056,-0.039293,0.012569,0.070970,-0.112187,-0.020383,1.0,1.0
2,2009-05-08 13:38:08+00:00,1.737480e+09,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08T13:38:08Z,2009-05-08,donald trump reads top ten financial tips on l...,13,4,0,38,...,0.098122,-0.128410,0.036980,0.003709,0.037471,0.052561,-0.012630,0.043487,1.0,1.0
3,2009-05-08 20:40:15+00:00,1.741161e+09,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08T20:40:15Z,2009-05-08,new blog post: celebrity apprentice finale and...,20,4,0,40,...,-0.078529,-0.028899,-0.093646,0.007925,0.095469,0.240477,-0.061366,-0.045210,1.0,1.0
4,2009-05-12 14:07:28+00:00,1.773561e+09,"""""""My persona will never be that of a wallflow...",2009-05-12T14:07:28Z,2009-05-12,"""""""my persona will never be that of a wallflow...",14,1,0,7,...,0.063643,0.019768,-0.022095,-0.148142,0.037149,-0.065220,0.079894,-0.006261,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46689,2021-01-06 21:17:24+00:00,1.346929e+18,https://t.co/Pm2PKV0Fp3,2021-01-06T21:17:24Z,2021-01-06,,21,2,0,17,...,0.023748,-0.017323,0.008929,-0.005868,-0.007515,0.021483,-0.006002,0.005868,1.0,6.0
46690,2021-01-06 23:01:04+00:00,1.346955e+18,These are the things and events that happen wh...,2021-01-06T23:01:04Z,2021-01-06,these are the things and events that happen wh...,23,2,0,1,...,-0.026220,0.126473,0.006688,-0.130505,-0.044662,0.097333,-0.080510,0.112731,1.0,6.0
46691,2021-01-08 00:10:24+00:00,1.347335e+18,https://t.co/csX07ZVWGe,2021-01-08T00:10:24Z,2021-01-08,,0,4,0,10,...,0.023748,-0.017323,0.008929,-0.005868,-0.007515,0.021483,-0.006002,0.005868,1.0,1.0
46692,2021-01-08 14:46:38+00:00,1.347555e+18,"The 75,000,000 great American Patriots who vot...",2021-01-08T14:46:38Z,2021-01-08,"the 75,000,000 great american patriots who vot...",14,4,0,46,...,0.102513,0.042074,0.029353,-0.151952,-0.137809,-0.050694,0.039298,-0.023867,1.0,1.0


In [59]:
df.to_csv(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\df.csv", sep=",", index=False, encoding="utf-8")

## Novelty Feature

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df["clean_text"])

df["novelty_score"] = np.asarray(X_tfidf.mean(axis=1)).ravel()

### seltene Begriffe

## Interaction Features

In [ ]:
df["sentiment_x_oil"] = df["sentiment_score"] * df["mentions_oil"]
df["emotion_x_caps"] = df["emotion_score"] * df["caps_ratio"]
df["geo_x_exclamation"] = df["mentions_geo"] * df["exclamation_count"]

## Final Feature Matrix

In [ ]:
feature_cols = [
    "sentiment_score",
    "emotion_score",
    "mentions_oil",
    "mentions_trade",
    "mentions_geo",
    "mentions_econ",
    "tweet_length",
    "word_count",
    "caps_ratio",
    "exclamation_count",
    "question_count",
    "tweets_last_1h",
    "tweets_last_6h",
    "novelty_score",
]

X = pd.concat([df[feature_cols], df.filter(like="emb_pca_")], axis=1)

## RoBERTA

Für Emotion Detection ist das beste Standardmodell:

👉 cardiffnlp/twitter-roberta-base-emotion

Es klassifiziert typischerweise in:

anger
joy
optimism
sadness

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import pandas as pd

### Modell laden

In [ ]:
model_name = "cardiffnlp/twitter-roberta-base-emotion"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

emotion_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True
)

### Emotion Feature Engineering

Wir extrahieren nicht nur Label, sondern vollständige Score-Verteilung → viel besser für ML.

In [ ]:
def get_emotion_scores(text):
    try:
        results = emotion_pipe(text[:512])[0]
        
        scores = {r["label"]: r["score"] for r in results}
        
        return pd.Series({
            "emotion_anger": scores.get("anger", 0),
            "emotion_joy": scores.get("joy", 0),
            "emotion_optimism": scores.get("optimism", 0),
            "emotion_sadness": scores.get("sadness", 0),
        })
    
    except:
        return pd.Series({
            "emotion_anger": 0,
            "emotion_joy": 0,
            "emotion_optimism": 0,
            "emotion_sadness": 0,
        })

### Apply auf DataFrame

In [ ]:
emotion_features = df["clean_text"].apply(get_emotion_scores)

df = pd.concat([df, emotion_features], axis=1)

### Emotion Features ableiten

In [ ]:
df["emotion_negative"] = df["emotion_anger"] + df["emotion_sadness"]
df["emotion_positive"] = df["emotion_joy"] + df["emotion_optimism"]

df["emotion_net"] = df["emotion_positive"] - df["emotion_negative"]

### Market-relevante Transformationsfeatures

In [ ]:
df["emotion_intensity"] = df[
    ["emotion_anger", "emotion_joy", "emotion_optimism", "emotion_sadness"]
].max(axis=1)

df["emotion_conflict"] = (
    df["emotion_positive"] * df["emotion_negative"]
)

Interpretation:

hoher emotion_conflict → ambivalente, unklare Tweets → oft noisy market signal

## Optional: Rolling Emotion Shock (sehr stark!)

In [ ]:
df = df.sort_values("timestamp")

df["emotion_net_ma_24h"] = df["emotion_net"].rolling("24h", on=df["timestamp"]).mean()
df["emotion_shock"] = df["emotion_net"] - df["emotion_net_ma_24h"]

Das ist extrem wichtig:

Märkte reagieren nicht auf Emotionen an sich
sondern auf Abweichungen vom Normalzustand

### Final Feature Set

In [ ]:
emotion_cols = [
    "emotion_anger",
    "emotion_joy",
    "emotion_optimism",
    "emotion_sadness",
    "emotion_negative",
    "emotion_positive",
    "emotion_net",
    "emotion_intensity",
    "emotion_conflict",
    "emotion_shock",
]